## Phase 1
Loading the model and inspecting the model's archeticture

In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, BertTokenizer, AutoTokenizer
import os
import shutil

model_name = "nomic-ai/nomic-embed-text-v1-unsupervised"
# model_name = "./nomic-custom-stack"
custom_tokens = [
    "cybersecurity", "kubernetes", "microservices", "hashmap", 
    "backpropagation", "asynchronous", "postgres", "minio", 
    "nextjs", "encapsulation", "javascript", "docker", "nginx", "ssh", "ddr4"
]

print("Loading baseline model and tokenizer...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)

# Phase 1: Architectural Inspection
vocab_size = len(tokenizer.vocab)
matrix_shape = model.embeddings.word_embeddings.weight.shape

print(f"Tokenizer Vocab Size: {vocab_size}")
print(f"Embedding Matrix Shape: {matrix_shape}")

C:\Users\I330\Desktop\Programming\sywa\embedding_surgery_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading baseline model and tokenizer...


<All keys matched successfully>


Tokenizer Vocab Size: 30522
Embedding Matrix Shape: torch.Size([30528, 768])


As we expected from the padding the inspection we notice the discrepancy: the vocabulary size is 30,522, but the embedding matrix has 30,528 rows.

Before I modify the tokenizer, I need to calculate the initialization vectors for my 15 custom tokens by averaging their existing subword embeddings.

In [2]:
old_embeddings = model.embeddings.word_embeddings.weight.data
hidden_size = old_embeddings.shape[1]

# Calculate initialization vectors using original subword splits
print("Calculating mathematically stable initialization vectors...")
init_vectors = []
for token in custom_tokens:
    subword_ids = tokenizer(token, add_special_tokens=False)["input_ids"]
    token_vector = old_embeddings[subword_ids].mean(dim=0)
    init_vectors.append(token_vector)
    
print(f"Calculated {len(init_vectors)} initialization vectors.")

Calculating mathematically stable initialization vectors...
Calculated 15 initialization vectors.


To understand how more about the model I will look into it's config

In [3]:
model.config

NomicBertConfig {
  "activation_function": "swiglu",
  "add_cross_attention": false,
  "architectures": [
    "NomicBertModel"
  ],
  "attn_pdrop": 0.0,
  "auto_map": {
    "AutoConfig": "nomic-ai/nomic-bert-2048--configuration_hf_nomic_bert.NomicBertConfig",
    "AutoModel": "nomic-ai/nomic-bert-2048--modeling_hf_nomic_bert.NomicBertModel",
    "AutoModelForMaskedLM": "nomic-ai/nomic-bert-2048--modeling_hf_nomic_bert.NomicBertForPreTraining"
  },
  "bos_token_id": null,
  "causal": false,
  "dense_seq_output": true,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": null,
  "fused_bias_fc": true,
  "fused_dropout_add_ln": true,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-12,
  "max_trained_positions": 2048,
  "mlp_fc1_bias": false,
  "mlp_fc2_bias": false,
  "model_type": "nomic_bert",
  "n_embd": 768,
  "n_head": 12,
  "n_inner": 3072,
  "n_layer": 12,
  "n_positions": 8192,
  "pad_token_id": null,
  "pad_vocab_size_multiple": 64,
  "parallel_block": false,
  "p

I also notice that the `pad_token_id` is set to `None` in the model metadata. This is a bug that will break batched attention masks later. I will patch it to 0 (the standard BERT padding token ID).

In [4]:
local_dir = "./nomic-custom-stack"
tokenizer.save_pretrained(local_dir)



# Wipe any stale state left over from a previous run before writing fresh files into it.
if os.path.exists(local_dir):
    shutil.rmtree(local_dir)
tokenizer.save_pretrained(local_dir)


id_to_token = {idx: tok for tok, idx in tokenizer.vocab.items()}
assert len(id_to_token) == 30522, f"Unexpected base vocab size: {len(id_to_token)}"
full_vocab_list = [id_to_token[i] for i in range(len(id_to_token))]
full_vocab_list.extend(custom_tokens)

vocab_path = os.path.join(local_dir, "vocab.txt")
with open(vocab_path, "w", encoding="utf-8") as f:
    f.write("\n".join(full_vocab_list) + "\n")

# Verify the write actually landed on disk before trusting
# it and moving on. Don't wait for a downstream KeyError to find out.
with open(vocab_path, "r", encoding="utf-8") as f:
    on_disk_lines = [line.strip() for line in f if line.strip()]
missing_from_disk = [t for t in custom_tokens if t not in on_disk_lines]
assert not missing_from_disk, f"Write failed, missing from vocab.txt on disk: {missing_from_disk}"
assert len(on_disk_lines) == 30522 + len(custom_tokens), (
    f"Expected {30522 + len(custom_tokens)} lines on disk, found {len(on_disk_lines)}"
)

# Here we are deleting it because it would read the tokenizer.json instead of our newly changed 
# vocab.txt file
tokenizer_json_path = os.path.join(local_dir, "tokenizer.json")
if os.path.exists(tokenizer_json_path):
    os.remove(tokenizer_json_path)
    print("Removed stale tokenizer.json so vocab.txt is authoritative.")

# Reload locally with never_split
modified_tokenizer = BertTokenizer.from_pretrained(
    local_dir,
    never_split=custom_tokens
)
new_vocab_count = len(modified_tokenizer.vocab)
print(f"Modified tokenizer loaded. New vocab size: {new_vocab_count}")
    
missing_from_tokenizer = [t for t in custom_tokens if t not in modified_tokenizer.vocab]
assert not missing_from_tokenizer, (
    f"Reload did not pick up new tokens: {missing_from_tokenizer}. "
    f"Vocab size is {new_vocab_count}, expected {30522 + len(custom_tokens)}."
)
assert new_vocab_count == 30522 + len(custom_tokens), (
    f"Vocab size mismatch: got {new_vocab_count}, expected {30522 + len(custom_tokens)}"
)

# The Configuration Patch
if getattr(model.config, "pad_token_id", None) is None:
    model.config.pad_token_id = 0
    print("Patched model config: pad_token_id set to 0")

if modified_tokenizer.pad_token_id is None:
    modified_tokenizer.pad_token_id = 0
    print("Patched tokenizer config: pad_token_id set to 0")

Removed stale tokenizer.json so vocab.txt is authoritative.
Modified tokenizer loaded. New vocab size: 30537
Patched model config: pad_token_id set to 0


Because I had to prepare the [MOSAIC](https://arxiv.org/pdf/2510.16797) research paper I had to learn more about Masked Language Modeling (MLM). So we know that we have to re-sync the PyTorch memory pointeers and preserve the model for future training

I am bypassing resize_token_embeddings() by instantiating a brand new nn.Embedding matrix of size 30,592, copying over the old weights, and injecting my calculated vectors.

In [5]:
#Physical matrix size: padded to a multiple of 64
new_matrix_rows = 30592

# Logical vocab size: the real number of known tokens (no padding)
new_logical_vocab_size = new_vocab_count  # 30522 + 15 = 30537

new_embedding_layer = nn.Embedding(new_matrix_rows, hidden_size)

# Copy the original 30,528 (padded) weights over
new_embedding_layer.weight.data[:30528] = old_embeddings

# Inject the 15 calculated vectors into their new slots (starting at 30522)
start_index = 30522
for i, vector in enumerate(init_vectors):
    target_index = start_index + i
    new_embedding_layer.weight.data[target_index] = vector

# Surgically swap the module
model.embeddings.word_embeddings = new_embedding_layer

# note: config.vocab_size must be the logical vocab count, NOT the padded
# matrix row count, matching the convention the original checkpoint used.
model.config.vocab_size = new_logical_vocab_size

# Force PyTorch to re-tie the weights in memory solves issues down the line with training using (MLM)
model.tie_weights()
print("Matrix expanded to 30,592 and weights tied successfully.")

Matrix expanded to 30,592 and weights tied successfully.


To verify I am going to make a string that utilizes several of the new domain tokens, and ensure they are appnedid in the index space, also do a test to gurantee no `NaN` or `INF` tesnor corruption happened

In [6]:
test_text = "Routing microservices with nginx and deploying nextjs and postgres on docker via ssh."
encoded = modified_tokenizer(test_text, return_tensors="pt")

print("Tokens mapping check:")
for token, token_id in zip(modified_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]), encoded["input_ids"][0].tolist()):
    print(f"{token:15} -> {token_id}")

# Verification Assertions
postgres_id = modified_tokenizer.vocab["postgres"]
assert postgres_id >= 30522, f"Surgery failed: 'postgres' mapped to {postgres_id}"

# Sanity check: config.vocab_size should equal the tokenizer's real vocab size,
# NOT the padded matrix size (30592).
assert model.config.vocab_size == len(modified_tokenizer.vocab), (
    f"config.vocab_size ({model.config.vocab_size}) does not match tokenizer "
    f"vocab ({len(modified_tokenizer.vocab)})"
)

model.eval()
with torch.no_grad():
    outputs = model(**encoded)

assert not torch.isnan(outputs.last_hidden_state).any(), "Forward pass produced NaNs!"
print("\nSuccess: Forward pass produced stable vectors. Output shape:", outputs.last_hidden_state.shape)

# Save final artifacts
model.save_pretrained(local_dir)
modified_tokenizer.save_pretrained(local_dir)
print(f"\nFinal model and tokenizer successfully saved to {local_dir}")

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Tokens mapping check:
[CLS]           -> 101
routing         -> 16972
microservices   -> 30524
with            -> 2007
nginx           -> 30534
and             -> 1998
deploy          -> 21296
##ing           -> 2075
nextjs          -> 30530
and             -> 1998
postgres        -> 30528
on              -> 2006
docker          -> 30533
via             -> 3081
ssh             -> 30535
.               -> 1012
[SEP]           -> 102

Success: Forward pass produced stable vectors. Output shape: torch.Size([1, 17, 768])


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.66it/s]


Final model and tokenizer successfully saved to ./nomic-custom-stack


In [7]:
local_model_name = local_dir
new_tokenizer = BertTokenizer.from_pretrained(local_model_name)
new_model = AutoModel.from_pretrained(local_model_name, trust_remote_code=True)
test_text = "Routing microservices with nginx and deploying nextjs and postgres on docker via ssh."
encoded = modified_tokenizer(test_text, return_tensors="pt")

print("Tokens mapping check:")
for token, token_id in zip(modified_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]), encoded["input_ids"][0].tolist()):
    print(f"{token:15} -> {token_id}")

# Verification Assertions
postgres_id = modified_tokenizer.vocab["postgres"]
assert postgres_id >= 30522, f"Surgery failed: 'postgres' mapped to {postgres_id}"

# Sanity check: config.vocab_size should equal the tokenizer's real vocab size,
# NOT the padded matrix size (30592).
assert model.config.vocab_size == len(modified_tokenizer.vocab), (
    f"config.vocab_size ({model.config.vocab_size}) does not match tokenizer "
    f"vocab ({len(modified_tokenizer.vocab)})"
)

model.eval()
with torch.no_grad():
    outputs = model(**encoded)

assert not torch.isnan(outputs.last_hidden_state).any(), "Forward pass produced NaNs!"
print("\nSuccess: Forward pass produced stable vectors. Output shape:", outputs.last_hidden_state.shape)

print("\n--- Rigorous Tensor Verification ---")
emb = model.embeddings.word_embeddings.weight.data
old_vocab_size = 30522  # real original vocabulary size (not padded)

# (1) UNCHANGED: compare only the real 30,522 tokens (ignore the 6 padding rows)
assert torch.equal(emb[:old_vocab_size], old_embeddings[:old_vocab_size]), "Old embeddings were modified!"
print(f"[OK] All {old_vocab_size} pretrained embeddings are bit-for-bit unchanged.")

# (2) CHANGED, correctly: every domain term is now ONE token
for tok in custom_tokens:
    pieces = modified_tokenizer.tokenize(tok)
    assert pieces == [tok], f"{tok} still splits: {pieces}"
print(f"[OK] All {len(custom_tokens)} domain terms now tokenize as a single token.")

# (3) ...and its embedding equals the mean of its old subword embeddings
for i, tok in enumerate(custom_tokens):
    new_id = modified_tokenizer.convert_tokens_to_ids(tok)
    expected = init_vectors[i]
    assert torch.equal(emb[new_id], expected), f"Bad init for {tok}"
print(f"[OK] All {len(custom_tokens)} new embeddings equal the mean of their old subword embeddings.")

<All keys matched successfully>


Tokens mapping check:
[CLS]           -> 101
routing         -> 16972
microservices   -> 30524
with            -> 2007
nginx           -> 30534
and             -> 1998
deploy          -> 21296
##ing           -> 2075
nextjs          -> 30530
and             -> 1998
postgres        -> 30528
on              -> 2006
docker          -> 30533
via             -> 3081
ssh             -> 30535
.               -> 1012
[SEP]           -> 102

Success: Forward pass produced stable vectors. Output shape: torch.Size([1, 17, 768])

--- Rigorous Tensor Verification ---
[OK] All 30522 pretrained embeddings are bit-for-bit unchanged.
[OK] All 15 domain terms now tokenize as a single token.
[OK] All 15 new embeddings equal the mean of their old subword embeddings.


In [8]:
PUSH = False  # change to True when ready to upload
REPO_ID = "Abood1521/cs-nomic-embed-text-v1-unsupervised"

if PUSH:
    model.push_to_hub(REPO_ID)
    modified_tokenizer.push_to_hub(REPO_ID)
    print(f"✅ Model and tokenizer published at: https://huggingface.co/{REPO_ID}")
else:
    print("Skipping upload (PUSH is False).")

C:\Users\I330\Desktop\Programming\sywa\embedding_surgery_2\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub\models--Abood1521--cs-nomic-embed-text-v1-unsupervised. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]
Pro

✅ Model and tokenizer published at: https://huggingface.co/Abood1521/cs-nomic-embed-text-v1-unsupervised
